# 01 — TF-IDF baseline search and final evaluation

CPU runtime is enough. This notebook compares logistic regression and linear SVM across three regularization settings on the same five development folds. It is self-contained.


Use these prepared files from the original project (the outputs of script 09):

```text
MyDrive/thesis_ml/data/dev.csv
MyDrive/thesis_ml/data/supplementare.csv
MyDrive/thesis_ml/data/test.csv        # needed only for final evaluation
```

The first two CSVs need `id`, `thread_id`, `testo`, `etichetta`; `dev.csv` also needs the existing `fold` column numbered 1–5. Labels are `strumentale`, `intrinseco`, `ambiguo`.

If you only have raw annotation files, first obtain the prepared CSVs from the original project; these notebooks do not guess a new split. No original Python scripts are needed once the CSVs are prepared.

Edit the configuration cell and run cells from top to bottom. Results are saved in Drive. Use one session at a time per experiment folder. Final evaluation is initially disabled so **Run all** cannot accidentally use the test set during selection. Enable it only after choosing the complete comparison protocol.

## Install dependencies
Run this cell first in a fresh runtime. The classifier library is pinned to preserve its API. Do not change package versions midway through a search.

In [ ]:
%pip install -q "scikit-learn==1.6.1" "pandas>=2.2,<3" "matplotlib>=3.9,<4" "joblib>=1.4,<2"

## Mount Drive and configure paths

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Edit PROJECT_DIR to the folder containing your data/ directory.
PROJECT_DIR = Path('/content/drive/MyDrive/thesis_ml')
DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_ROOT = PROJECT_DIR / 'results'
C_VALUES = [0.1, 1.0, 10.0]
FAMILIES = ['logistic_regression', 'linear_svm']
EXPECTED_DEV_ROWS = 641  # Set None only if you intentionally use a different dataset.
EXPECTED_TEST_ROWS = 159
FINAL_SEED = 42
RUN_FINAL = False  # Set True only after you have frozen your full comparison protocol.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Helpers and baseline models
TF-IDF converts text into numeric features. `C` controls the classifier: smaller values give stronger regularization, larger values give weaker regularization. The vectorizer is fitted only on each fold’s training examples.

In [ ]:
from pathlib import Path
import gc, hashlib, importlib.metadata, json, marshal, os, platform, random, re, time, types, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from IPython.display import display

LABELS = ['strumentale', 'intrinseco', 'ambiguo']
LABEL2ID = {name: i for i, name in enumerate(LABELS)}
TEST_NOTE = ('The existing test set was already used in preliminary experiments. '
             'It is not a retrospectively untouched holdout.')

def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def safe_name(value):
    # Hugging Face IDs can contain a slash; local checkpoints can have absolute paths.
    return re.sub(r'[^A-Za-z0-9_.-]', '_', str(value)).strip('._')[:80] + '_' + json_hash(str(value))[:8]

def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))
    os.replace(temporary, path)

def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)

def read_table(path, with_folds=False, allow_empty=False):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Copy the prepared CSV files from the original project.')
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    needed = {'id', 'thread_id', 'testo', 'etichetta'} | ({'fold'} if with_folds else set())
    if not needed.issubset(frame.columns):
        raise ValueError(f'{path.name}: missing columns {sorted(needed - set(frame.columns))}')
    if frame.empty and not allow_empty:
        raise ValueError(f'{path.name} is empty.')
    if frame['id'].duplicated().any():
        raise ValueError(f'{path.name}: duplicate example IDs.')
    for col in ['id', 'thread_id', 'testo']:
        if frame[col].str.strip().eq('').any():
            raise ValueError(f'{path.name}: blank {col}.')
    if not set(frame['etichetta']).issubset(LABELS):
        raise ValueError(f'{path.name}: unknown labels {set(frame.etichetta) - set(LABELS)}')
    if with_folds:
        numeric = pd.to_numeric(frame['fold'], errors='raise')
        if (numeric % 1 != 0).any():
            raise ValueError('Fold numbers must be integers.')
        frame['fold'] = numeric.astype(int)
        if set(frame.fold) != set(range(1, 6)):
            raise ValueError('Expected the five existing folds numbered 1 to 5.')
        if (frame.groupby('thread_id')['fold'].nunique() > 1).any():
            raise ValueError('A thread appears in more than one fold.')
    return frame

def load_development(data_dir):
    dev = read_table(data_dir / 'dev.csv', with_folds=True)
    supp = read_table(data_dir / 'supplementare.csv', allow_empty=True)
    if EXPECTED_DEV_ROWS is not None and len(dev) != EXPECTED_DEV_ROWS:
        raise ValueError(f'Expected {EXPECTED_DEV_ROWS} development rows, found {len(dev)}.')
    if set(dev.id) & set(supp.id):
        raise ValueError('Development and supplementary example IDs overlap.')
    return dev, supp

def fold_data(dev, supp, fold):
    validation = dev[dev.fold == fold].reset_index(drop=True)
    gold_train = dev[dev.fold != fold].drop(columns='fold')
    eligible = supp[~supp.thread_id.isin(set(validation.thread_id))]
    training = pd.concat([gold_train, eligible], ignore_index=True)
    if set(training.thread_id) & set(validation.thread_id):
        raise ValueError('Training/validation thread overlap.')
    if set(training.etichetta) != set(LABELS):
        raise ValueError('A training fold is missing a class.')
    return training, validation

def metrics_for(truth, predictions):
    report = classification_report(truth, predictions, labels=LABELS,
                                   output_dict=True, zero_division=0)
    return {'macro_f1': float(report['macro avg']['f1-score']),
            'accuracy': float(np.mean(np.asarray(truth) == np.asarray(predictions))),
            'per_class': {label: report[label] for label in LABELS}}

def code_hash(functions):
    # Normalize notebook cell filenames and line positions, preserving executable code.
    def normalize(c):
        constants = tuple(normalize(x) if isinstance(x, types.CodeType) else x for x in c.co_consts)
        line_key = 'co_linetable' if hasattr(c, 'co_linetable') else 'co_lnotab'
        return c.replace(co_filename='', co_firstlineno=1, co_consts=constants, **{line_key: b''})
    return json_hash({f.__name__: hashlib.sha256(marshal.dumps(normalize(f.__code__))).hexdigest()
                      for f in functions})

def experiment_manifest(protocol, functions, extra_versions=None):
    versions = {name: importlib.metadata.version(name)
                for name in ['numpy', 'pandas', 'scikit-learn', 'matplotlib']}
    versions.update(extra_versions or {})
    return {'workflow_version': 1, 'protocol': protocol,
            'data_hashes': {name: file_hash(DATA_DIR / name)
                            for name in ['dev.csv', 'supplementare.csv']},
            'code_hash': code_hash([read_table, load_development, fold_data, metrics_for] + functions),
            'python': platform.python_version(), 'versions': versions}

def start_experiment(manifest, output_root, prefix):
    experiment_id = json_hash(manifest)[:16]
    directory = output_root / f'{prefix}_{experiment_id}'
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / 'manifest.json'
    if path.exists() and json.loads(path.read_text()) != manifest:
        raise ValueError('Manifest mismatch. Use a different experiment folder.')
    atomic_json(path, manifest)
    return directory

def assert_inputs_unchanged():
    for name, expected in MANIFEST['data_hashes'].items():
        if file_hash(DATA_DIR / name) != expected:
            raise ValueError(f'{name} changed during this session. Rerun setup before continuing.')

def read_completed(directory, job, expected_epochs):
    path = directory / 'metrics.json'
    if not path.exists():
        return None
    result = json.loads(path.read_text())
    if result.get('job') != job:
        raise ValueError(f'Job mismatch at {directory}.')
    if result.get('status') != 'complete':
        return None  # Partial runs restart from their original pretrained weights.
    prediction_file = directory / 'predictions.csv'
    if (len(result['epochs']) != expected_epochs or
        not prediction_file.exists() or
        file_hash(prediction_file) != result.get('predictions_hash')):
        raise ValueError(f'Incomplete/corrupt completed result at {directory}.')
    return result

def save_completed(directory, job, epoch_metrics, predictions, elapsed, extra=None):
    directory.mkdir(parents=True, exist_ok=True)
    atomic_csv(directory / 'predictions.csv', predictions)
    value = {'status': 'complete', 'job': job, 'epochs': epoch_metrics,
             'seconds': float(elapsed), 'predictions_hash': file_hash(directory / 'predictions.csv')}
    value.update(extra or {})
    atomic_json(directory / 'metrics.json', value)
    return value

def checked_test(dev, supp):
    # Only call in the final evaluation section, after selection is frozen.
    test = read_table(DATA_DIR / 'test.csv')
    if EXPECTED_TEST_ROWS is not None and len(test) != EXPECTED_TEST_ROWS:
        raise ValueError(f'Expected {EXPECTED_TEST_ROWS} test rows, found {len(test)}.')
    training = pd.concat([dev.drop(columns='fold'), supp], ignore_index=True)
    if set(training.id) & set(test.id) or set(training.thread_id) & set(test.thread_id):
        raise ValueError('Final training/test overlap in IDs or threads.')
    if set(training.etichetta) != set(LABELS):
        raise ValueError('Final training is missing a class.')
    return training, test

def final_identity(selected, test):
    return {'manifest_hash': json_hash(MANIFEST), 'selected': selected,
            'test_hash': file_hash(DATA_DIR / 'test.csv'), 'seed': FINAL_SEED,
            'test_rows': len(test), 'test_note': TEST_NOTE}

def show_confusion(truth, prediction, path):
    matrix = confusion_matrix(truth, prediction, labels=LABELS)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(matrix, cmap='Blues')
    ax.set_xticks(range(3), LABELS, rotation=20)
    ax.set_yticks(range(3), LABELS)
    ax.set(xlabel='Predicted', ylabel='True')
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center')
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    display(fig)
    plt.close(fig)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline
import joblib

def new_vectorizer():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000,
                          stop_words='english', sublinear_tf=True)

def new_classifier(family, C):
    if family == 'logistic_regression':
        return LogisticRegression(C=C, max_iter=10000, class_weight='balanced',
                                  solver='lbfgs', random_state=FINAL_SEED)
    if family == 'linear_svm':
        return LinearSVC(C=C, max_iter=10000, class_weight='balanced', dual=True,
                         random_state=FINAL_SEED)
    raise ValueError(f'Unknown baseline {family}')

def fit_baseline(family, C, X, y):
    model = new_classifier(family, C)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        model.fit(X, y)
    convergence = [str(w.message) for w in caught if issubclass(w.category, ConvergenceWarning)]
    if convergence:
        raise RuntimeError('Classifier did not converge. Increase max_iter and rerun as a new experiment: '
                           + '; '.join(convergence))
    return model

def baseline_job(family, C, fold):
    return {'family': family, 'C': float(C), 'fold': int(fold), 'seed': FINAL_SEED}

def baseline_path(family, C, fold):
    return EXP_DIR / 'runs' / family / f'C_{C:g}' / f'fold_{fold}'

def baseline_summary():
    rows, selected = [], {}
    for family in FAMILIES:
        candidates = []
        for C in C_VALUES:
            scores = []
            for fold in range(1, 6):
                result = read_completed(baseline_path(family, C, fold),
                                        baseline_job(family, C, fold), 1)
                if result is not None:
                    scores.append(result['epochs'][0]['macro_f1'])
            if len(scores) == 5:
                row = {'family': family, 'C': float(C), 'mean_macro_f1': float(np.mean(scores)),
                       'fold_std': float(np.std(scores, ddof=1)), 'fold_scores': scores}
                rows.append(row)
                candidates.append(row)
        if len(candidates) == len(C_VALUES):
            selected[family] = sorted(candidates, key=lambda r: (-r['mean_macro_f1'], r['C']))[0]
    return pd.DataFrame(rows), selected

## Check development data and initialize experiment
Only development and supplementary CSVs are read here. Existing fold assignments are reused. A changed configuration, code, input or environment creates a separate experiment folder.

In [ ]:
if not C_VALUES or len(set(C_VALUES)) != len(C_VALUES) or any(c <= 0 for c in C_VALUES):
    raise ValueError('C_VALUES must contain distinct positive values.')
if not FAMILIES or len(set(FAMILIES)) != len(FAMILIES) or not set(FAMILIES).issubset({'logistic_regression', 'linear_svm'}):
    raise ValueError('Choose unique supported baseline families.')
dev, supp = load_development(DATA_DIR)
MANIFEST = experiment_manifest(
    {'kind': 'baselines', 'families': FAMILIES, 'C_values': C_VALUES, 'labels': LABELS,
     'folds': [1, 2, 3, 4, 5], 'seed': FINAL_SEED,
     'selection': 'mean_fold_macro_f1; exact ties prefer smaller C'},
    [new_vectorizer, new_classifier, fit_baseline, baseline_job, baseline_summary])
EXP_DIR = start_experiment(MANIFEST, OUTPUT_ROOT, 'baselines')
print('Results:', EXP_DIR)
print(f'{len(FAMILIES) * len(C_VALUES) * 5} fits; test.csv is not loaded.')
display(dev.groupby(['fold', 'etichetta']).size().unstack(fill_value=0))

## Run the search
This is 30 small classifier fits. Completed folds are saved and skipped on rerun. A convergence failure stops selection rather than silently accepting a poorly fitted candidate.

In [ ]:
assert_inputs_unchanged()
for fold in range(1, 6):
    training, validation = fold_data(dev, supp, fold)
    pending = [(family, C) for family in FAMILIES for C in C_VALUES
               if read_completed(baseline_path(family, C, fold), baseline_job(family, C, fold), 1) is None]
    if not pending:
        print(f'Fold {fold}: already complete.')
        continue
    vectorizer = new_vectorizer()
    X_train = vectorizer.fit_transform(training.testo)
    X_validation = vectorizer.transform(validation.testo)
    for family, C in pending:
        started = time.time()
        model = fit_baseline(family, C, X_train, training.etichetta)
        prediction = model.predict(X_validation)
        score = metrics_for(validation.etichetta.tolist(), prediction.tolist())
        score['epoch'] = 1
        pred_frame = validation[['id', 'thread_id', 'etichetta']].rename(columns={'etichetta': 'true'})
        pred_frame['predicted'] = prediction
        save_completed(baseline_path(family, C, fold), baseline_job(family, C, fold),
                       [score], pred_frame, time.time() - started,
                       {'training_rows': len(training), 'validation_rows': len(validation)})
        print(f'{family}, C={C:g}, fold={fold}: macro-F1={score["macro_f1"]:.4f}')

## Compare candidates and select the baseline
Choose each family’s `C` by mean fold macro-F1. Exact ties prefer smaller `C`. The best baseline is chosen from those two families. Fold standard deviations are descriptive spreads, not confidence intervals.

In [ ]:
assert_inputs_unchanged()
summary, selected = baseline_summary()
if len(selected) != len(FAMILIES):
    raise RuntimeError('Search is incomplete; run the search cell again before selecting.')
atomic_csv(EXP_DIR / 'candidate_summary.csv', summary)
selection_document = {'manifest_hash': json_hash(MANIFEST), 'data_hashes': MANIFEST['data_hashes'],
                      'selected': selected,
                      'overall_baseline': sorted(selected.values(),
                          key=lambda r: (-r['mean_macro_f1'], FAMILIES.index(r['family'])))[0],
                      'note': 'CV scores were used for selection and are development estimates.'}
atomic_json(EXP_DIR / 'selected_configs.json', selection_document)
display(summary.drop(columns='fold_scores').sort_values(['family', 'mean_macro_f1'], ascending=[True, False]))
print('Selected:', json.dumps(selected, indent=2))
fig, ax = plt.subplots(figsize=(7, 4))
for family in FAMILIES:
    subset = summary[summary.family == family].sort_values('C')
    ax.errorbar(subset.C, subset.mean_macro_f1, yerr=subset.fold_std, marker='o', label=family)
ax.set(xscale='log', xlabel='C (larger = weaker regularization)', ylabel='Mean validation macro-F1')
ax.legend()
fig.tight_layout()
fig.savefig(EXP_DIR / 'regularization_comparison.png', dpi=180)
display(fig)
plt.close(fig)

## Final training and test evaluation — optional
After completing both notebooks’ development searches, freeze the comparison: best baseline plus best configuration of each transformer. Set `RUN_FINAL=True` in the configuration cell and rerun this section. The chosen baseline is fitted on development plus supplementary data and evaluated on the separate test. The original test was already used in preliminary experiments; preserve that limitation in the thesis. Do not return to tuning based on its scores.

In [ ]:
if not RUN_FINAL:
    print('Final training skipped. Set RUN_FINAL=True after freezing the comparison protocol.')
else:
    assert_inputs_unchanged()
    saved = json.loads((EXP_DIR / 'selected_configs.json').read_text())
    if saved['manifest_hash'] != json_hash(MANIFEST):
        raise ValueError('Selection belongs to another experiment.')
    training, test = checked_test(dev, supp)
    selected_baseline = saved['overall_baseline']
    directory = EXP_DIR / 'final' / selected_baseline['family']
    identity = final_identity(selected_baseline, test)
    result_path = directory / 'evaluation.json'
    if result_path.exists():
        previous = json.loads(result_path.read_text())
        if previous['identity'] != identity:
            raise ValueError('Existing final evaluation has a different configuration or test dataset.')
        print('Final evaluation already complete:', result_path)
        display(pd.DataFrame(previous['report']).T)
    else:
        directory.mkdir(parents=True, exist_ok=True)
        vectorizer = new_vectorizer()
        X_train = vectorizer.fit_transform(training.testo)
        classifier = fit_baseline(selected_baseline['family'], selected_baseline['C'],
                                  X_train, training.etichetta)
        prediction = classifier.predict(vectorizer.transform(test.testo))
        pipeline = Pipeline([('tfidf', vectorizer), ('classifier', classifier)])
        joblib.dump(pipeline, directory / 'model.joblib')
        report = classification_report(test.etichetta, prediction, labels=LABELS,
                                       output_dict=True, zero_division=0)
        predictions = test[['id', 'thread_id', 'etichetta']].rename(columns={'etichetta': 'true'})
        predictions['predicted'] = prediction
        atomic_csv(directory / 'test_predictions.csv', predictions)
        show_confusion(test.etichetta, prediction, directory / 'confusion_matrix.png')
        atomic_json(result_path, {'identity': identity, 'report': report,
                                 'training_rows': len(training), 'test_rows': len(test)})
        display(pd.DataFrame(report).T)
        print('Saved:', directory)

## Saved artifacts
The experiment folder contains the manifest, fold predictions, candidate table, selected configurations and plot. Final evaluation adds `model.joblib`, test predictions, metrics and a confusion matrix. For new-text inference: `joblib.load(path_to_model).predict(list_of_texts)`.